In [1]:
import pandas as pd 
import numpy as np
import joblib
import time
import sklearn.metrics
from xgboost import XGBClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier

labels = [0,1,2,3,4,5,6,7,8,9,10,11]


## Load dataset generated by CovaS

In [5]:
print("Loading train...")
train_df = pd.read_csv('./DS/iec104_train.csv', low_memory=False)
print("Loading test...")
test_df= pd.read_csv('./DS/iec104_test.csv', low_memory=False)

FS = ['flow total IEC104_U_Message packets', 'bw packet APDU length max', 'cot=1', 'type_id_process_information_in_monitor_direction', 'total flow packets', 'type_id_process_information_in_control_direction', 'bw packet APDU length std', 'flow packets APDU total length', 'type_id_system_information_in_control_direction', 'bw total IEC104_U_Message packets', 'flow packet APDU length std', 'bw total IEC104_S_Message packets', 'flow down/up ratio', 'cot=3', 'flow iec104 packts/s', 'flow packet APDU length max', 'flow packet APDU length mean', 'flow IAT min', 'bw IAT min', 'bw packets APDU total length', 'flow IAT tot', 'flow IAT max', 'fw packet APDU length std', 'bw iec104 packts/s', 'init fw window bytes', 'flow total IEC104_I_Message_SingleIOA packets', 'flow active time std', 'flow idle time std', 'bw packet APDU length mean', 'flow total IEC104_S_Message packets', 'fw iec104 packts/s', 'fw TCP total header length', 'flow IAT std', 'fw IAT max', 'fw IAT std', 'fw_subflow_bytes', 'bw IAT std', 'bw_subflow_packets', 'flow idle time mean', 'fw total IEC104_I_Message_SingleIOA packets', 'flow active time min', 'fw iec104 bytes/s', 'cot=10', 'flow idle time max', 'bw iec104 bytes/s', 'bw IAT max', 'flow active time mean', 'flow IAT mean', 'flow active time max', 'total fw packets', 'fw iAT tot', 'bw IAT tot', 'flow idle time min', 'flow iec104 bytes/s', 'bw IAT mean', 'fw IAT min', 'fw IAT mean', 'bw avg bytes/bulk', 'init bw window bytes', 'bw avg packets/bulk', 'fw total IEC104_U_Message packets', 'bw_subflow_bytes']

# X_train = train_df.drop(['Label'], axis=1)
X_train = train_df[FS]
y_train = train_df['Label']
# X_test = test_df.drop(['Label'], axis=1)
X_test = test_df[FS]
y_test = test_df['Label']

Loading train...
Loading test...


In [12]:
from imblearn.combine import SMOTEENN
smote = SMOTEENN(random_state=42)
print(X_train.shape)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)
X_aug = np.concatenate([X_train, X_resampled], axis=0)
y_aug = np.concatenate([y_train, y_resampled], axis=0)
X_aug = pd.DataFrame(X_aug, columns=X_train.columns)

print(X_aug.shape)

(4800, 62)
(6391, 62)


In [13]:
xgb_params = {
    'max_depth': 128,
    'n_estimators': 5000,
    'learning_rate': 0.1, # best 0.5;  0.01 0.05
    'tree_method': 'gpu_hist',         
    'predictor': 'gpu_predictor',  
    'eval_metric': 'auc',
    'booster': 'gbtree',
    'objective':"multi:softprob",     
    'random_state': 42,
    'early_stopping_rounds': 20,
}
# xgb_params = {'n_estimators': 2184, 'learning_rate': 0.0532267325712996, 'max_depth': 10, 'min_child_weight': 3, 'gamma': 0.23533968844918063, 'subsample': 0.8128088333277587, 'colsample_bytree': 0.8753774433336527, 'reg_alpha': 0.9996275523662602, 'reg_lambda': 2.970083284239074, 'tree_method': 'gpu_hist',         
#     'predictor': 'gpu_predictor',  
#     'eval_metric': 'auc',
#     'booster': 'gbtree',
#     'objective':"multi:softprob",     
#     'random_state': 42,
#     'early_stopping_rounds': 20,}

print("XGBClassifier Starting")
xgb_model = XGBClassifier(**xgb_params)
xgb_model.fit(
    X_aug, y_aug,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_pred = xgb_model.predict(X_test)
print("XGBClassifier Finished")

print("Accuracy:", sklearn.metrics.accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision:", sklearn.metrics.precision_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("DR:", sklearn.metrics.recall_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("F1:", sklearn.metrics.f1_score(y_true=y_test, y_pred=y_pred, average='macro'))
labels = sorted(y_test.unique())
cm = sklearn.metrics.confusion_matrix(y_test, y_pred)
print("     " + " ".join(f"{lbl:6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{lbl:3d} " + " ".join(f"{val:6d}" for val in row))

XGBClassifier Starting
XGBClassifier Finished
Accuracy: 0.8476331360946746
Precision: 0.843190747118927
DR: 0.8476331360946746
F1: 0.8422014021885772
          0      1      2      3      4      5      6      7      8      9     10     11
  0    168      0      0      0      0      0      1      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      0    169      0      0      0      0      0      0      0      0      0
  3      0      0      0    168      0      1      0      0      0      0      0      0
  4      0      0      0      0    169      0      0      0      0      0      0      0
  5      0      0      0      0      1    168      0      0      0      0      0      0
  6      0      0      0      0      0      0    169      0      0      0      0      0
  7      0      1     40      0      0      0      0     66     43      7     12      0
  8      0      0      0      0      0      0      0     

## XGBoost

In [4]:
xgb_params = {
    'max_depth': 128,
    'n_estimators': 5000,
    'learning_rate': 0.1, # best 0.5;  0.01 0.05
    'tree_method': 'gpu_hist',         
    'predictor': 'gpu_predictor',  
    'eval_metric': 'auc',
    'booster': 'gbtree',
    'objective':"multi:softprob",     
    'random_state': 42,
    'early_stopping_rounds': 20,
}
# xgb_params = {'n_estimators': 2184, 'learning_rate': 0.0532267325712996, 'max_depth': 10, 'min_child_weight': 3, 'gamma': 0.23533968844918063, 'subsample': 0.8128088333277587, 'colsample_bytree': 0.8753774433336527, 'reg_alpha': 0.9996275523662602, 'reg_lambda': 2.970083284239074, 'tree_method': 'gpu_hist',         
#     'predictor': 'gpu_predictor',  
#     'eval_metric': 'auc',
#     'booster': 'gbtree',
#     'objective':"multi:softprob",     
#     'random_state': 42,
#     'early_stopping_rounds': 20,}

print("XGBClassifier Starting")
xgb_model = XGBClassifier(**xgb_params)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_pred = xgb_model.predict(X_test)
print("XGBClassifier Finished")

print("Accuracy:", sklearn.metrics.accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision:", sklearn.metrics.precision_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("DR:", sklearn.metrics.recall_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("F1:", sklearn.metrics.f1_score(y_true=y_test, y_pred=y_pred, average='macro'))
labels = sorted(y_test.unique())
cm = sklearn.metrics.confusion_matrix(y_test, y_pred)
print("     " + " ".join(f"{lbl:6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{lbl:3d} " + " ".join(f"{val:6d}" for val in row))

XGBClassifier Starting
XGBClassifier Finished
Accuracy: 0.8515779092702169
Precision: 0.8460157742611553
DR: 0.851577909270217
F1: 0.8456321910580735
          0      1      2      3      4      5      6      7      8      9     10     11
  0    167      0      0      2      0      0      0      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      0    169      0      0      0      0      0      0      0      0      0
  3      0      0      0    169      0      0      0      0      0      0      0      0
  4      0      0      0      0    169      0      0      0      0      0      0      0
  5      0      0      0      0      0    169      0      0      0      0      0      0
  6      0      0      0      0      0      0    169      0      0      0      0      0
  7      0      1     42      3      3      0      2     66     33      4     15      0
  8      0      0      0      0      0      0      0     

In [14]:
joblib.dump(xgb_model, './models/raw_xgb.pkl')

['./models/raw_xgb.pkl']

## ExtraTree

In [15]:
et_params = {
    "n_estimators": 73,
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy"
}

print("ExtraTreesClassifier Starting")
et_model = ExtraTreesClassifier(**et_params)
et_model.fit(X=X_train, y=y_train)
et_start_time = time.time()
y_pred = et_model.predict(X_test)
et_end_time = time.time()
et_time = et_end_time - et_start_time
print("ExtraTreesClassifier Finished")
print("ExtraTrees Time:", et_end_time - et_start_time)
print("Accuracy:", sklearn.metrics.accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision:", sklearn.metrics.precision_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("DR:", sklearn.metrics.recall_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("F1:", sklearn.metrics.f1_score(y_true=y_test, y_pred=y_pred, average='macro'))
labels = sorted(y_test.unique())
cm = sklearn.metrics.confusion_matrix(y_test, y_pred)
print("     " + " ".join(f"{lbl:6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{lbl:3d} " + " ".join(f"{val:6d}" for val in row))

ExtraTreesClassifier Starting
ExtraTreesClassifier Finished
ExtraTrees Time: 0.09052467346191406
Accuracy: 0.8515779092702169
Precision: 0.8497541367784608
DR: 0.851577909270217
F1: 0.8474413076979164
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      0    169      0      0      0      0      0      0      0      0      0
  3      0      0      0    169      0      0      0      0      0      0      0      0
  4      0      0      0      0    169      0      0      0      0      0      0      0
  5      0      0      0      0      0    169      0      0      0      0      0      0
  6      0      0      0      0      0      0    169      0      0      0      0      0
  7      0      1     31      0      0      0      0     79     34     11     13      0
  8   

In [89]:
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn import metrics as sklearn

# === Hàm phụ (nếu đã có thì bỏ qua) ===
def calculate_macro_tpr_fpr(cm):
    C = cm.shape[0]
    tprs, fprs = [], []
    for c in range(C):
        TP = cm[c, c]
        FN = cm[c, :].sum() - TP
        FP = cm[:, c].sum() - TP
        TN = cm.sum() - TP - FN - FP
        tpr = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        fpr = FP / (FP + TN) if (FP + TN) > 0 else 0.0
        tprs.append(tpr)
        fprs.append(fpr)
    return float(np.mean(tprs)), float(np.mean(fprs))

# === Tham số cố định ===
et_params = {
    "n_estimators": 70,        # sẽ thay trong vòng lặp
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy"
}

START, END, STEP = 60, 90, 1  # có thể đổi STEP=1 để quét mịn

print("ExtraTreesClassifier Sweep Starting")

records = []
best_key = None
best_model, best_cm, best_pred = None, None, None
best_n, best_time = None, None

for n in range(START, END + 1, STEP):
    params = et_params.copy()
    params["n_estimators"] = n

    model = ExtraTreesClassifier(**params)
    model.fit(X_train, y_train)

    t0 = time.time()
    pred = model.predict(X_test)
    t1 = time.time()

    acc = sklearn.accuracy_score(y_test, pred)
    precision = sklearn.precision_score(y_test, pred, average='macro', zero_division=0)
    f1 = sklearn.f1_score(y_test, pred, average='macro')
    recall = sklearn.recall_score(y_test, pred, average='macro')
    cm = sklearn.confusion_matrix(y_test, pred)
    et_fp = cm[0, 1] if cm.shape == (2, 2) else None
    tpr, fpr = calculate_macro_tpr_fpr(cm)

    records.append({
        "n_estimators": n,
        "time_sec": t1 - t0,
        "accuracy": acc,
        "precision_macro": precision,
        "f1_macro": f1,
        "recall_macro": recall,
        "macro_TPR": tpr,
        "macro_FPR": fpr,
        "fp_(cm01_if_binary)": et_fp
    })

    key = (acc, f1, recall, -n)  # tie-break
    if best_key is None or key > best_key:
        best_key = key
        best_model, best_cm, best_pred = model, cm, pred
        best_n, best_time = n, t1 - t0

print("ExtraTreesClassifier Sweep Finished")

# === Tổng hợp kết quả ===
df_results = pd.DataFrame(records).sort_values(
    by=["accuracy", "f1_macro", "recall_macro", "n_estimators"], 
    ascending=[False, False, False, True]
).reset_index(drop=True)

print("\n=== Top-10 cấu hình theo Accuracy ===")
print(df_results.head(10)[["n_estimators", "accuracy", "f1_macro", "recall_macro", "time_sec"]])

# === Report tốt nhất ===
et_acc = sklearn.accuracy_score(y_test, best_pred)
et_precision = sklearn.precision_score(y_test, best_pred, average='macro', zero_division=0)
et_f1 = sklearn.f1_score(y_test, best_pred, average='macro')
et_recall = sklearn.recall_score(y_test, best_pred, average='macro')
et_cm = best_cm
et_fp = et_cm[0, 1] if et_cm.shape == (2, 2) else None
et_tpr, et_fpr = calculate_macro_tpr_fpr(et_cm)

print("\n===== Best ExtraTrees report =====")
print(f"Best n_estimators: {best_n}")
print("ExtraTrees Time:", best_time)
print("ExtraTrees Accuracy:", et_acc)
print("ExtraTrees Precision:", et_precision)
print("ExtraTrees F1:", et_f1)
print("ExtraTrees Recall:", et_recall)
print("ExtraTrees FP:", et_fp)
print("ExtraTrees CM:\n", et_cm)
print(f'ExtraTrees Macro-average TPR: {et_tpr}')
print(f'ExtraTrees Macro-average FPR: {et_fpr}')

# df_results.to_csv("./et_sweep_n_estimators_30_500.csv", index=False)


ExtraTreesClassifier Sweep Starting
ExtraTreesClassifier Sweep Finished

=== Top-10 cấu hình theo Accuracy ===
   n_estimators  accuracy  f1_macro  recall_macro  time_sec
0            73  0.866371  0.863021      0.866371  0.113818
1            72  0.865385  0.861985      0.865385  0.122401
2            70  0.865385  0.861837      0.865385  0.112467
3            63  0.864892  0.861809      0.864892  0.114389
4            69  0.864892  0.861701      0.864892  0.117126
5            74  0.864892  0.861369      0.864892  0.114547
6            65  0.864398  0.861188      0.864398  0.113499
7            61  0.864398  0.861037      0.864398  0.118825
8            71  0.863905  0.861014      0.863905  0.116514
9            67  0.863905  0.860611      0.863905  0.115421

===== Best ExtraTrees report =====
Best n_estimators: 73
ExtraTrees Time: 0.11381816864013672
ExtraTrees Accuracy: 0.866370808678501
ExtraTrees Precision: 0.8660786473330858
ExtraTrees F1: 0.8630207588759967
ExtraTrees Recall: 0

In [17]:
joblib.dump(et_model, './models/raw_et.pkl')

['./models/raw_et.pkl']

## RandomForest

In [16]:
rf_params = {
    "n_estimators": 69,
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy"
}

print("RandomForestClassifier Starting")
rf_model = RandomForestClassifier(**rf_params)
rf_model.fit(X=X_train, y=y_train)
rf_start_time = time.time()
y_pred = rf_model.predict(X_test)
rf_end_time = time.time()
rf_time = rf_end_time - rf_start_time
print("RandomForestClassifier Finished")
print("RandomForest Time:", rf_end_time - rf_start_time)
print("Accuracy:", sklearn.metrics.accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision:", sklearn.metrics.precision_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("DR:", sklearn.metrics.recall_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("F1:", sklearn.metrics.f1_score(y_true=y_test, y_pred=y_pred, average='macro'))
labels = sorted(y_test.unique())
cm = sklearn.metrics.confusion_matrix(y_test, y_pred)
print("     " + " ".join(f"{lbl:6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{lbl:3d} " + " ".join(f"{val:6d}" for val in row))

RandomForestClassifier Starting
RandomForestClassifier Finished
RandomForest Time: 0.07588887214660645
Accuracy: 0.8515779092702169
Precision: 0.8484217833970585
DR: 0.8515779092702171
F1: 0.8462798169528903
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      0    169      0      0      0      0      0      0      0      0      0
  3      0      0      0    169      0      0      0      0      0      0      0      0
  4      0      0      0      0    169      0      0      0      0      0      0      0
  5      0      0      0      0      1    168      0      0      0      0      0      0
  6      0      0      0      0      0      0    169      0      0      0      0      0
  7      0      1     37      0      2      0      2     70     34     10     13      0

In [4]:
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics as sklearn

# === Hàm phụ (nếu bạn đã có thì bỏ qua) ===
def calculate_macro_tpr_fpr(cm):
    """
    cm: confusion_matrix (C x C)
    Trả về (macro_TPR, macro_FPR)
    """
    C = cm.shape[0]
    tprs, fprs = [], []
    for c in range(C):
        TP = cm[c, c]
        FN = cm[c, :].sum() - TP
        FP = cm[:, c].sum() - TP
        TN = cm.sum() - TP - FN - FP
        tpr = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        fpr = FP / (FP + TN) if (FP + TN) > 0 else 0.0
        tprs.append(tpr)
        fprs.append(fpr)
    return float(np.mean(tprs)), float(np.mean(fprs))

# === Tham số cố định ===
rf_params = {
    "n_estimators": 60,        # sẽ bị thay mỗi vòng lặp
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy"
}

START, END, STEP = 50, 90, 1   # có thể đổi STEP=1 nếu muốn quét mịn hơn

print("RandomForestClassifier Sweep Starting")

records = []
best_key = None  # (acc, f1, recall, -n_estimators) để có tie-break rõ ràng
best_model = None
best_cm = None
best_pred = None
best_n = None
best_time = None

for n in range(START, END + 1, STEP):
    params = rf_params.copy()
    params["n_estimators"] = n

    model = RandomForestClassifier(**params)
    model.fit(X=X_train, y=y_train)

    t0 = time.time()
    pred = model.predict(X_test)
    t1 = time.time()

    acc = sklearn.accuracy_score(y_true=y_test, y_pred=pred)
    precision = sklearn.precision_score(y_true=y_test, y_pred=pred, average='macro', zero_division=0)
    f1 = sklearn.f1_score(y_true=y_test, y_pred=pred, average='macro')
    recall = sklearn.recall_score(y_true=y_test, y_pred=pred, average='macro')
    cm = sklearn.confusion_matrix(y_true=y_test, y_pred=pred)
    # Lưu ý: rf_fp = cm[0, 1] chỉ có ý nghĩa khi bài toán nhị phân và label 0/1 nằm đúng trục
    rf_fp = cm[0, 1] if cm.shape == (2, 2) else None
    tpr, fpr = calculate_macro_tpr_fpr(cm)

    records.append({
        "n_estimators": n,
        "time_sec": t1 - t0,
        "accuracy": acc,
        "precision_macro": precision,
        "f1_macro": f1,
        "recall_macro": recall,
        "macro_TPR": tpr,
        "macro_FPR": fpr,
        "fp_(cm01_if_binary)": rf_fp
    })

    # Tie-break: ưu tiên acc -> f1 -> recall; nếu vẫn hòa, chọn n_estimators nhỏ hơn
    key = (acc, f1, recall, -n)
    if (best_key is None) or (key > best_key):
        best_key = key
        best_model = model
        best_cm = cm
        best_pred = pred
        best_n = n
        best_time = t1 - t0

print("RandomForestClassifier Sweep Finished")

# === Tổng hợp kết quả thành DataFrame và in Top-10 theo Accuracy ===
df_results = pd.DataFrame(records).sort_values(
    by=["accuracy", "f1_macro", "recall_macro", "n_estimators"], ascending=[False, False, False, True]
).reset_index(drop=True)

print("\n=== Top-10 cấu hình theo Accuracy ===")
print(df_results.head(10)[["n_estimators", "accuracy", "f1_macro", "recall_macro", "time_sec"]])

# === In report mô hình tốt nhất (giữ format bạn đang dùng) ===
rf_acc = sklearn.accuracy_score(y_true=y_test, y_pred=best_pred)
rf_precision = sklearn.precision_score(y_true=y_test, y_pred=best_pred, average='macro', zero_division=0)
rf_f1 = sklearn.f1_score(y_true=y_test, y_pred=best_pred, average='macro')
rf_recall = sklearn.recall_score(y_true=y_test, y_pred=best_pred, average='macro')
rf_cm = best_cm
rf_fp = rf_cm[0, 1] if rf_cm.shape == (2, 2) else None
rf_tpr, rf_fpr = calculate_macro_tpr_fpr(rf_cm)

print("\n===== Best RandomForest report =====")
print(f"Best n_estimators: {best_n}")
print("RandomForest Time:", best_time)
print("RandomForest Accuracy:", rf_acc)
print("RandomForest Precision:", rf_precision)
print("RandomForest F1:", rf_f1)
print("RandomForest Recall:", rf_recall)
print("RandomForest FP:", rf_fp)
print("RandomForest CM:", rf_cm)
print(f'RandomForest Macro-average TPR: {rf_tpr}')
print(f'RandomForest Macro-average FPR: {rf_fpr}')

RandomForestClassifier Sweep Starting
RandomForestClassifier Sweep Finished

=== Top-10 cấu hình theo Accuracy ===
   n_estimators  accuracy  f1_macro  recall_macro  time_sec
0            69  0.867357  0.864635      0.867357  0.125913
1            67  0.867357  0.864400      0.867357  0.118462
2            75  0.866864  0.863942      0.866864  0.127681
3            55  0.866864  0.863498      0.866864  0.122875
4            71  0.866371  0.863459      0.866371  0.127537
5            89  0.865878  0.863037      0.865878  0.124729
6            61  0.865878  0.862846      0.865878  0.124759
7            65  0.865878  0.862843      0.865878  0.122364
8            85  0.865385  0.862569      0.865385  0.132442
9            63  0.865385  0.862547      0.865385  0.194182

===== Best RandomForest report =====
Best n_estimators: 69
RandomForest Time: 0.12591290473937988
RandomForest Accuracy: 0.8673570019723866
RandomForest Precision: 0.8681013507240944
RandomForest F1: 0.8646349705114257
Rando

In [18]:
joblib.dump(rf_model, './models/raw_rf.pkl')

['./models/raw_rf.pkl']

In [4]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    f1_score,
    recall_score,
    confusion_matrix,
    roc_auc_score
)

## GBM

In [19]:
from lightgbm import LGBMClassifier, early_stopping

lgbm_params = {
 "boosting_type": "gbdt",
    "objective": "multiclass",
    "num_class": len(y_train.unique()),
    "metric": "multi_logloss",

    "learning_rate": 0.13,
    "n_estimators": 5000,
    "num_leaves": 192,
    "max_depth": -1,

    "min_child_samples": 48,
    "reg_alpha": 0.15,
    "reg_lambda": 0.8,
    "feature_fraction": 0.75,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,

    "device": "cpu",
    "verbosity": -1,
    "random_state": 42,
    "n_jobs": -1, 
}

# Train LightGBM Model
print("LightGBM Starting")
lgbm_model = LGBMClassifier(**lgbm_params)
lgbm_model.fit(
    X=X_train, y=y_train, 
    eval_set=[(X_test, y_test)],
    callbacks=[early_stopping(stopping_rounds=30)],
)
y_pred = lgbm_model.predict(X_test)
print("LightGBM Finished")
print("Accuracy:", sklearn.metrics.accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision:", sklearn.metrics.precision_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("DR:", sklearn.metrics.recall_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("F1:", sklearn.metrics.f1_score(y_true=y_test, y_pred=y_pred, average='macro'))
labels = sorted(y_test.unique())
cm = sklearn.metrics.confusion_matrix(y_test, y_pred)
print("     " + " ".join(f"{lbl:6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{lbl:3d} " + " ".join(f"{val:6d}" for val in row))

LightGBM Starting
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[70]	valid_0's multi_logloss: 0.44605
LightGBM Finished
Accuracy: 0.8412228796844181
Precision: 0.8325986013928018
DR: 0.8412228796844182
F1: 0.8337193091178179
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      0    169      0      0      0      0      0      0      0      0      0
  3      0      0      0    169      0      0      0      0      0      0      0      0
  4      0      0      0      1    168      0      0      0      0      0      0      0
  5      0      0      0      0      1    168      0      0      0      0      0      0
  6      0      0      0      0      0      0    169      0      0      0      0      0
  7      0      1   

In [ ]:
joblib.dump(lgbm_model, './models/raw_lgbm.pkl')

['./models/raw_lgbm.pkl']

## catboost

In [33]:
from catboost import CatBoostClassifier
import time, sklearn

cat_params = {
    # "iterations": 5000,
    # "depth": 15,
    "learning_rate": 0.8,
    "loss_function": "MultiClass",
    "task_type": "GPU",
    "random_seed": 42,
    "eval_metric": "MultiClass",      # macro F1 cho đa lớp
    "od_type": "Iter",             # early stopping kiểu số vòng
    "od_wait": 30,                 # tương đương early_stopping_rounds=30
    "classes_count": len(y_train.unique())
}

print("CatBoost Starting")
cat_model = CatBoostClassifier(**cat_params)

# fit model
cat_model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test), use_best_model=True,
    verbose=False
)

cat_start_time = time.time()
y_pred = cat_model.predict(X_test)
cat_end_time = time.time()
cat_time = cat_end_time - cat_start_time
print("CatBoost Finished")
print("CatBoost Time:", cat_time)
print("Accuracy:", sklearn.metrics.accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision:", sklearn.metrics.precision_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("DR:", sklearn.metrics.recall_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("F1:", sklearn.metrics.f1_score(y_true=y_test, y_pred=y_pred, average='macro'))
labels = sorted(y_test.unique())
cm = sklearn.metrics.confusion_matrix(y_test, y_pred)
print("     " + " ".join(f"{lbl:6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{lbl:3d} " + " ".join(f"{val:6d}" for val in row))

CatBoost Starting
CatBoost Finished
CatBoost Time: 0.0128021240234375
Accuracy: 0.841715976331361
Precision: 0.8336735775970916
DR: 0.841715976331361
F1: 0.8348235283204054
          0      1      2      3      4      5      6      7      8      9     10     11
  0    168      0      1      0      0      0      0      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      0    169      0      0      0      0      0      0      0      0      0
  3      0      0      0    167      1      0      1      0      0      0      0      0
  4      0      0      0      3    165      0      1      0      0      0      0      0
  5      0      0      0      0      0    168      0      0      0      0      1      0
  6      0      0      0      1      2      0    166      0      0      0      0      0
  7      0      1     33      3      6      1      4     79     23      9     10      0
  8      0      0      0      0   

In [34]:
joblib.dump(cat_model, './models/raw_cat.pkl')

['./models/raw_cat.pkl']

## biLSTM

In [75]:
# ============================
# LSTM cho Tabular — IEC/ICS preset — KHÔNG focus lớp nào
#  - Stratified split (VAL 15%)
#  - StandardScaler
#  - DataLoader (shuffle, không sampler/boost)
#  - LSTM (2 tầng, BiLSTM) + AttnPooling
#  - Mixup công bằng cho mọi lớp
#  - EMA (float-only) device-safe
#  - Early-stopping theo Macro-F1 (VAL)
#  - Đánh giá: Acc, Macro-F1, per-class metrics
# ============================
import math, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, precision_recall_fscore_support

# ----- Device & seed -----
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42); np.random.seed(42)

# ===== Dữ liệu =====
X_full = train_df[FS].values
y_full = train_df['Label'].values
X_test = test_df[FS].values
y_test = test_df["Label"].values

num_classes  = int(max(y_full.max(), y_test.max())) + 1
num_features = X_full.shape[1]

# ===== 1) Stratified split: train_in / val (15%) =====
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(sss.split(X_full, y_full))
X_tr_in, y_tr_in = X_full[train_idx], y_full[train_idx]
X_val,   y_val   = X_full[val_idx],   y_full[val_idx]

# ===== 2) Scaler =====
scaler = StandardScaler()
X_tr_in_sc = scaler.fit_transform(X_tr_in)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

# ===== 3) DataLoaders =====
Xtr_t = torch.tensor(X_tr_in_sc, dtype=torch.float32)
ytr_t = torch.tensor(y_tr_in,    dtype=torch.long)
Xva_t = torch.tensor(X_val_sc,   dtype=torch.float32)
yva_t = torch.tensor(y_val,      dtype=torch.long)
Xte_t = torch.tensor(X_test_sc,  dtype=torch.float32)
yte_t = torch.tensor(y_test,     dtype=torch.long)

batch_size = 2048
train_loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch_size, shuffle=True,  pin_memory=True)
val_loader   = DataLoader(TensorDataset(Xva_t, yva_t), batch_size=4096, shuffle=False, pin_memory=True)
test_loader  = DataLoader(TensorDataset(Xte_t, yte_t), batch_size=4096, shuffle=False, pin_memory=True)

# ===== 4) Tiện ích: chia feature thành "chuỗi" =====
# Ta coi features như một chuỗi gồm S steps, mỗi step có input_size = STEP_DIM
# (pad 0 nếu F không chia hết)
STEP_DIM = 16  # bạn có thể thử 8/16/32
def as_sequence(x_np: np.ndarray, step_dim: int = STEP_DIM):
    F = x_np.shape[1]
    S = int(np.ceil(F / step_dim))
    pad = S * step_dim - F
    if pad > 0:
        x_np = np.pad(x_np, ((0,0),(0,pad)), mode="constant")
    return x_np.reshape(-1, S, step_dim), S

# ===== 5) Mô hình LSTM + AttnPooling =====
class AttnPool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.proj = nn.Linear(d, 1)
    def forward(self, H):        # H: [B,S,D]
        score = self.proj(H).squeeze(-1)           # [B,S]
        w = torch.softmax(score, dim=1)            # [B,S]
        pooled = (H * w.unsqueeze(-1)).sum(1)      # [B,D]
        return pooled

class LSTMTabular(nn.Module):
    def __init__(self, step_dim, hidden=256, layers=2, n_classes=10, dropout=0.2, bidir=True):
        super().__init__()
        self.step_dim = step_dim
        self.in_norm  = nn.LayerNorm(step_dim)
        self.lstm = nn.LSTM(
            input_size=step_dim,
            hidden_size=hidden,
            num_layers=layers,
            batch_first=True,
            dropout=dropout if layers > 1 else 0.0,
            bidirectional=bidir
        )
        d_out = hidden * (2 if bidir else 1)
        self.pool = AttnPool(d_out)
        self.head = nn.Sequential(
            nn.Linear(d_out, d_out//2), nn.ReLU(), nn.Dropout(0.20),
            nn.Linear(d_out//2, n_classes)
        )
    def forward(self, x):                       # x: [B, F]
        B, F = x.shape
        S = int(math.ceil(F / self.step_dim))
        pad = S * self.step_dim - F
        if pad > 0:
            x = torch.nn.functional.pad(x, (0, pad), value=0.0)
        x = x.view(B, S, self.step_dim)         # [B,S,step_dim]
        x = self.in_norm(x)
        H, _ = self.lstm(x)                     # [B,S,D]
        z = self.pool(H)                        # [B,D]
        return self.head(z)                     # [B,C]

model = LSTMTabular(step_dim=STEP_DIM, hidden=256, layers=2, n_classes=num_classes, dropout=0.15, bidir=True).to(device)

# ===== 6) Loss/optim/scheduler + mixup (công bằng) =====
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=2e-4)

total_epochs = 120
warmup_epochs = 19
steps_per_epoch = max(1, len(train_loader))
def lr_lambda(step):
    warmup_steps = warmup_epochs * steps_per_epoch
    total_steps  = total_epochs * steps_per_epoch
    if step < warmup_steps: return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, (total_steps - warmup_steps))
    return 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def mixup_pair(x, y, alpha=0.10):
    if alpha <= 0: return x, y, 1.0, None
    lam  = np.random.beta(alpha, alpha)
    perm = torch.randperm(x.size(0), device=x.device)
    x_m  = lam * x + (1 - lam) * x[perm]
    y_perm = y[perm]
    return x_m, y, lam, y_perm

# ===== 7) EMA (float-only) — device-safe =====
ema_decay = 0.995
ema_state = {}
for k, v in model.state_dict().items():
    if v.is_floating_point():
        ema_state[k] = v.detach().clone().to(device=v.device, dtype=v.dtype)

def ema_update(model, ema_state, decay: float):
    with torch.no_grad():
        for k, v in model.state_dict().items():
            if not v.is_floating_point(): continue
            if (ema_state[k].device != v.device) or (ema_state[k].dtype != v.dtype):
                ema_state[k] = ema_state[k].to(device=v.device, dtype=v.dtype)
            ema_state[k].mul_((decay)).add_(v.detach(), alpha=1.0 - decay)

def load_ema_state(model, ema_state):
    current = model.state_dict(); merged = {}
    for k, v in current.items():
        merged[k] = ema_state.get(k, v).to(device=v.device, dtype=v.dtype)
    model.load_state_dict(merged, strict=True)

# ===== 8) Train + Early-stopping theo Macro-F1 (VAL) =====
best_macro_f1, best_ema_snapshot, wait, patience = -1.0, None, 0, 20
global_step = 0

for epoch in range(total_epochs):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        xb_m, ya, lam, yb_ = mixup_pair(xb, yb, alpha=0.10)
        logits = model(xb_m)
        loss = lam * criterion(logits, ya) + (1 - lam) * criterion(logits, yb_) if yb_ is not None else criterion(logits, ya)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        optimizer.step(); scheduler.step()
        ema_update(model, ema_state, ema_decay)
        global_step += 1

    # Eval VAL với EMA
    saved_live = {k: v.detach().clone() for k, v in model.state_dict().items()}
    load_ema_state(model, ema_state)
    model.eval()
    with torch.no_grad():
        logits_val = []
        for xb, _ in val_loader:
            logits_val.append(model(xb.to(device)).cpu())
        logits_val = torch.cat(logits_val, 0).numpy()
    pred_val = logits_val.argmax(1)
    macro_f1 = f1_score(y_val, pred_val, average='macro', zero_division=0)

    if macro_f1 > best_macro_f1:
        best_macro_f1, wait = macro_f1, 0
        best_ema_snapshot = {k: v.detach().cpu().clone() for k, v in ema_state.items()}
    else:
        wait += 1

    model.load_state_dict(saved_live, strict=True)
    if wait >= patience:
        print(f"[Early-stop] epoch {epoch+1} | best VAL Macro-F1 = {best_macro_f1:.4f}")
        break

# Nạp EMA tốt nhất
if best_ema_snapshot is not None:
    ema_state = {k: t.clone().to(device) for k, t in best_ema_snapshot.items()}
    load_ema_state(model, ema_state)
model.eval()

# ===== 9) SUY LUẬN TEST =====
with torch.no_grad():
    logits_test = []
    for xb, _ in test_loader:
        logits_test.append(model(xb.to(device)).cpu())
    logits_test = torch.cat(logits_test, 0).numpy()

y_pred = logits_test.argmax(1)
def softmax_np(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

# Xác suất dự đoán cho LSTM
lstm_preds = softmax_np(logits_test)   # logits_test là numpy array từ bước suy luận LSTM
lstm_prediction = y_pred 

# ===== 10) Đánh giá =====
print("Accuracy:", sklearn.metrics.accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision:", sklearn.metrics.precision_score(y_true=y_test, y_pred=y_pred, average='macro',zero_division=0))
print("DR:", sklearn.metrics.recall_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("F1:", sklearn.metrics.f1_score(y_true=y_test, y_pred=y_pred, average='macro'))
cm = sklearn.metrics.confusion_matrix(y_test, y_pred)
print("     " + " ".join(f"{lbl:6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{lbl:3d} " + " ".join(f"{val:6d}" for val in row))

Accuracy: 0.6494082840236687
Precision: 0.6268185248798771
DR: 0.6494082840236686
F1: 0.6247997587187769
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    130     39      0      0      0      0      0      0      0      0      0
  2      0     54     96      0      0      0      0     18      0      0      0      1
  3      0      0      0      0     87     22     60      0      0      0      0      0
  4      0      0      0      0     94     20     54      0      0      0      0      1
  5      0      0      0      0     69    100      0      0      0      0      0      0
  6      0      0      0      0     61     10     98      0      0      0      0      0
  7      0      0      9      0      0      0      0     94     66      0      0      0
  8      0      0      0      0      0      0      0      4    165      0      0      0
  9      0    

In [77]:
torch.save({
    "model_state": model.state_dict(),
    "ema_state": ema_state,
    "scaler": scaler,
    "config": {
        "step_dim": STEP_DIM,
        "num_classes": num_classes,
        "num_features": num_features,
    }
}, './models/raw_lstm.pkl')

# resDNN

In [70]:
import math, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, precision_recall_fscore_support

# ===== CỜ TUỲ CHỌN =====
ENABLE_HEAD_FINETUNE = True      # finetune riêng head vài epoch để nhạy với 7/10
ENABLE_OVR_BLEND     = False     # OvR cho 7/10 (LogReg) rồi blend vào xác suất DNN
OPT_BIAS_FOR_TARGETS = False     # True: tối ưu macro-F1 chỉ trên {7,10} khi chọn T & bias

# ===== Device & seed =====
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42); np.random.seed(42)

# ===== Dữ liệu =====
X_full = train_df[FS].values
y_full = train_df['Label'].values
X_test = test_df[FS].values
y_test = test_df["Label"].values

num_classes  = int(max(y_full.max(), y_test.max())) + 1
num_features = X_full.shape[1]

# ===== 1) Stratified split: train_in / val (15%) =====
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(sss.split(X_full, y_full))
X_tr_in, y_tr_in = X_full[train_idx], y_full[train_idx]
X_val,   y_val   = X_full[val_idx],   y_full[val_idx]

# ===== 2) Scaler =====
scaler = StandardScaler()
X_tr_in_sc = scaler.fit_transform(X_tr_in)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

# ===== 3) Sampler cân bằng + boost 7/10 =====
counts = np.bincount(y_tr_in, minlength=num_classes).astype(np.float32)
inv = counts.sum() / (counts + 1e-9)
inv /= inv.mean()
boost = np.ones(num_classes, dtype=np.float32)
boost[[7,10]] = 1.6  # tăng tần suất gặp 7/10 (giữ hệ số)
sample_w = (inv * boost)[y_tr_in]
sampler  = WeightedRandomSampler(weights=sample_w, num_samples=len(sample_w), replacement=True)

# ===== 4) DataLoaders =====
Xtr_t = torch.tensor(X_tr_in_sc, dtype=torch.float32)
ytr_t = torch.tensor(y_tr_in,    dtype=torch.long)
Xva_t = torch.tensor(X_val_sc,   dtype=torch.float32)
yva_t = torch.tensor(y_val,      dtype=torch.long)
Xte_t = torch.tensor(X_test_sc,  dtype=torch.float32)
yte_t = torch.tensor(y_test,     dtype=torch.long)

batch_size = 1024
train_loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch_size, sampler=sampler, pin_memory=True)
val_loader   = DataLoader(TensorDataset(Xva_t, yva_t), batch_size=4096, shuffle=False, pin_memory=True)
test_loader  = DataLoader(TensorDataset(Xte_t, yte_t), batch_size=4096, shuffle=False, pin_memory=True)

# ===== 5) Mô hình =====
class ResidualBlock(nn.Module):
    def __init__(self, d_in, d_hid, p=0.25):
        super().__init__()
        self.lin1 = nn.Linear(d_in, d_hid)
        self.bn1  = nn.BatchNorm1d(d_hid)
        self.lin2 = nn.Linear(d_hid, d_in)
        self.ln2  = nn.LayerNorm(d_in)
        self.drop = nn.Dropout(p)
    def forward(self, x):
        h = self.drop(torch.relu(self.bn1(self.lin1(x))))
        h = self.lin2(h)
        return torch.relu(self.ln2(x + h))

class ResDNN(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        W = 512
        self.stem = nn.Sequential(nn.Linear(in_dim, W), nn.BatchNorm1d(W), nn.ReLU(), nn.Dropout(0.30))
        self.block1 = ResidualBlock(W, W//2, p=0.30)
        self.block2 = ResidualBlock(W, W//2, p=0.25)
        self.block3 = ResidualBlock(W, W//2, p=0.20)
        self.head   = nn.Linear(W, n_classes)
    def forward(self, x):
        h = self.stem(x)
        h = self.block1(h); h = self.block2(h); h = self.block3(h)
        return self.head(h)

model = ResDNN(num_features, num_classes).to(device)

# ===== 6) CB-Focal Loss + class weights (Effective Number) =====
def cb_class_weights(counts, beta=0.999):
    eff = (1.0 - beta) / (1.0 - np.power(beta, counts + 1e-12))
    w   = eff / eff.mean()
    return torch.tensor(w, dtype=torch.float32, device=device)

class FocalCE(nn.Module):
    def __init__(self, gamma=1.7, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight
    def forward(self, logits, target):
        logp = torch.log_softmax(logits, dim=1)
        p    = torch.exp(logp)
        idx  = torch.arange(logits.size(0), device=logits.device)
        pt   = p[idx, target]
        loss = -((1-pt) ** self.gamma) * logp[idx, target]
        if self.weight is not None:
            loss = loss * self.weight[target]
        return loss.mean()

cls_w  = cb_class_weights(counts, beta=0.999)
criterion = FocalCE(gamma=1.7, weight=cls_w)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=2e-4)

# ===== 7) Lịch LR: warmup + cosine =====
total_epochs  = 120
warmup_epochs = 8
steps_per_epoch = max(1, len(train_loader))
def lr_lambda(current_step):
    warmup_steps = warmup_epochs * steps_per_epoch
    total_steps  = total_epochs * steps_per_epoch
    if current_step < warmup_steps:
        return (current_step + 1) / warmup_steps
    progress = (current_step - warmup_steps) / max(1, (total_steps - warmup_steps))
    return 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ===== 8) Mixup (giảm khi batch có 7/10) =====
def mixup_dynamic(x, y, base_alpha=0.10):
    alpha = base_alpha
    if any(lbl in (7,10) for lbl in y.tolist()):
        alpha = 0.02
    if alpha <= 0: return x, y, 1.0, None
    lam  = np.random.beta(alpha, alpha)
    perm = torch.randperm(x.size(0), device=x.device)
    x_m  = lam * x + (1 - lam) * x[perm]
    y_perm = y[perm]
    return x_m, y, lam, y_perm

# ===== 9) EMA (float-only) — DEVICE SAFE =====
ema_decay = 0.995
ema_state = {}
for k, v in model.state_dict().items():
    if v.is_floating_point():
        ema_state[k] = v.detach().clone().to(device=v.device, dtype=v.dtype)

def ema_update(model, ema_state, decay: float):
    with torch.no_grad():
        for k, v in model.state_dict().items():
            if not v.is_floating_point():
                continue
            if (ema_state[k].device != v.device) or (ema_state[k].dtype != v.dtype):
                ema_state[k] = ema_state[k].to(device=v.device, dtype=v.dtype)
            ema_state[k].mul_(decay).add_(v.detach(), alpha=1.0 - decay)

def load_ema_state(model, ema_state):
    current = model.state_dict()
    merged = {}
    for k, v in current.items():
        if k in ema_state:
            merged[k] = ema_state[k].to(device=v.device, dtype=v.dtype)
        else:
            merged[k] = v
    model.load_state_dict(merged, strict=True)

# ===== 10) Train + Early-stopping Macro-F1 (VAL) =====
best_f1, best_ema_snapshot, wait, patience = -1.0, None, 0, 20
global_step = 0

for epoch in range(total_epochs):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        xb_m, ya, lam, yb_ = mixup_dynamic(xb, yb, base_alpha=0.10)
        logits = model(xb_m)
        loss = lam * criterion(logits, ya) + (1 - lam) * criterion(logits, yb_) if yb_ is not None else criterion(logits, ya)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        optimizer.step()
        scheduler.step()
        ema_update(model, ema_state, ema_decay)
        global_step += 1

    # --- Eval VAL với EMA ---
    saved_live = {k: v.detach().clone() for k, v in model.state_dict().items()}
    load_ema_state(model, ema_state)
    model.eval()
    with torch.no_grad():
        logits_val = []
        for xb, _ in val_loader:
            logits_val.append(model(xb.to(device)).cpu())
        logits_val = torch.cat(logits_val, 0).numpy()
    pred_val = logits_val.argmax(1)
    f1_val_all = f1_score(y_val, pred_val, average='macro')
    f1_val_target = f1_score(y_val, pred_val, labels=[7,10], average='macro', zero_division=0)
    f1_val = f1_val_target if OPT_BIAS_FOR_TARGETS else f1_val_all

    if f1_val > best_f1:
        best_f1, wait = f1_val, 0
        best_ema_snapshot = {k: v.detach().cpu().clone() for k, v in ema_state.items()}
    else:
        wait += 1

    model.load_state_dict(saved_live, strict=True)

    if wait >= patience:
        print(f"[Early-stop] epoch {epoch+1} | best VAL Macro-F1 ({'targets' if OPT_BIAS_FOR_TARGETS else 'all'}) = {best_f1:.4f}")
        break

# ===== 11) Load EMA tốt nhất và (tuỳ chọn) finetune head =====
if best_ema_snapshot is not None:
    ema_state = {k: t.clone().to(device) for k, t in best_ema_snapshot.items()}
    load_ema_state(model, ema_state)

if ENABLE_HEAD_FINETUNE:
    for p in model.stem.parameters():   p.requires_grad = False
    for p in model.block1.parameters(): p.requires_grad = False
    for p in model.block2.parameters(): p.requires_grad = False
    for p in model.block3.parameters(): p.requires_grad = False
    optimizer = torch.optim.AdamW(model.head.parameters(), lr=5e-4, weight_decay=1e-4)
    ft_epochs, ft_patience, ft_wait, best_ft = 12, 6, 0, -1.0
    for epoch in range(ft_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)  # tắt mixup khi finetune head
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.head.parameters(), 3.0)
            optimizer.step()
            ema_update(model, ema_state, ema_decay)
        # eval VAL
        saved_live = {k: v.detach().clone() for k, v in model.state_dict().items()}
        load_ema_state(model, ema_state)
        model.eval()
        with torch.no_grad():
            logits_val = []
            for xb, _ in val_loader:
                logits_val.append(model(xb.to(device)).cpu())
            logits_val = torch.cat(logits_val, 0).numpy()
        pred_val = logits_val.argmax(1)
        f1_val_all = f1_score(y_val, pred_val, average='macro')
        f1_val_target = f1_score(y_val, pred_val, labels=[7,10], average='macro', zero_division=0)
        f1_val = f1_val_target if OPT_BIAS_FOR_TARGETS else f1_val_all
        if f1_val > best_ft:
            best_ft, ft_wait = f1_val, 0
            best_ema_snapshot = {k: v.detach().cpu().clone() for k, v in ema_state.items()}
        else:
            ft_wait += 1
        model.load_state_dict(saved_live, strict=True)
        if ft_wait >= ft_patience:
            print(f"[Finetune head Early-stop] epoch {epoch+1} | best VAL Macro-F1 = {best_ft:.4f}")
            break
    if best_ema_snapshot is not None:
        ema_state = {k: t.clone().to(device) for k, t in best_ema_snapshot.items()}
        load_ema_state(model, ema_state)

model.eval()

# ===== 12) Temperature scaling (VAL) =====
def softmax_np(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z); return e / e.sum(axis=1, keepdims=True)

with torch.no_grad():
    logits_val = []
    for xb, _ in val_loader:
        logits_val.append(model(xb.to(device)).cpu())
    logits_val = torch.cat(logits_val, 0).numpy()
    logits_test = []
    for xb, _ in test_loader:
        logits_test.append(model(xb.to(device)).cpu())
    logits_test = torch.cat(logits_test, 0).numpy()

best_T, best_val_f1 = 1.0, -1.0
for T in np.arange(0.5, 3.05, 0.1):
    p_val = softmax_np(logits_val / T)
    pred  = p_val.argmax(1)
    f1_all    = f1_score(y_val, pred, average='macro')
    f1_target = f1_score(y_val, pred, labels=[7,10], average='macro', zero_division=0)
    f1_use    = f1_target if OPT_BIAS_FOR_TARGETS else f1_all
    if f1_use > best_val_f1:
        best_val_f1, best_T = f1_use, float(T)

# ===== 13) (Tuỳ chọn) OvR blend cho 7/10 =====
if ENABLE_OVR_BLEND:
    from sklearn.linear_model import LogisticRegression
    X_ovr = X_tr_in_sc; y_ovr = y_tr_in
    ovr_targets = [7,10]
    lr_models = {}
    for t in ovr_targets:
        y_bin = (y_ovr == t).astype(int)
        lr = LogisticRegression(max_iter=200)
        lr.fit(X_ovr, y_bin)
        lr_models[t] = lr
    ovr_scores_test = {t: lr_models[t].predict_proba(X_test_sc)[:,1] for t in ovr_targets}

# ===== 14) Logit-bias search trên VAL để đẩy 7/10 =====
targets = [7,10]
grid = np.arange(0.0, 1.30, 0.10)
best_b, best_f1_bias = np.zeros(num_classes), -1.0

for b7 in grid:
    for b10 in grid:
        b = np.zeros(num_classes, dtype=np.float32)
        b[7], b[10] = b7, b10
        p_val = softmax_np((logits_val / best_T) + b[None, :])
        pred = p_val.argmax(1)
        f1_all    = f1_score(y_val, pred, average='macro')
        f1_target = f1_score(y_val, pred, labels=targets, average='macro', zero_division=0)
        f1_use    = f1_target if OPT_BIAS_FOR_TARGETS else f1_all
        if f1_use > best_f1_bias:
            best_f1_bias, best_b = f1_use, b.copy()

# ===== 15) Suy luận TEST với T* và bias* (và OvR nếu bật) =====
p_test = softmax_np((logits_test / best_T) + best_b[None, :])

if ENABLE_OVR_BLEND:
    for t in [7,10]:
        s = ovr_scores_test[t]  # [N]
        p_test[:, t] *= (0.5 + 0.5 * s)
    p_test = p_test / p_test.sum(axis=1, keepdims=True)

y_pred = p_test.argmax(1)
dnn_preds = p_test                                   # shape: (n_samples, n_classes)
dnn_prediction = y_pred
# ===== 16) Đánh giá =====
print("Accuracy:", sklearn.metrics.accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision:", sklearn.metrics.precision_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("DR:", sklearn.metrics.recall_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("F1:", sklearn.metrics.f1_score(y_true=y_test, y_pred=y_pred, average='macro'))
# labels = sorted(y_test.unique())
cm = sklearn.metrics.confusion_matrix(y_test, y_pred)
print("     " + " ".join(f"{lbl:6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{lbl:3d} " + " ".join(f"{val:6d}" for val in row))

[Finetune head Early-stop] epoch 12 | best VAL Macro-F1 = 0.6566
Accuracy: 0.7095660749506904
Precision: 0.7142270674131085
DR: 0.7095660749506902
F1: 0.6976033663197122
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    139     30      0      0      0      0      0      0      0      0      0
  2      0     18    129      0      0      0      0     22      0      0      0      0
  3      0      0      0     41     24     38     66      0      0      0      0      0
  4      0      0      0     20     55     32     62      0      0      0      0      0
  5      0      0      0      4     11    153      1      0      0      0      0      0
  6      0      0      0      6     28     19    116      0      0      0      0      0
  7      0      1      0      0      0      0      0    134     34      0      0      0
  8      0      0      0      0      

In [71]:
torch.save({
    "model_state": model.state_dict(),   # trọng số cuối (đã nạp EMA tốt nhất)
    "ema_state": ema_state,              # lưu lại EMA state để khôi phục
    "scaler": scaler,                    # StandardScaler để inference sau này
    "config": {
        "num_classes": num_classes,
        "num_features": num_features,
        "enable_head_finetune": ENABLE_HEAD_FINETUNE,
        "opt_bias_for_targets": OPT_BIAS_FOR_TARGETS,
        "enable_ovr_blend": ENABLE_OVR_BLEND,
    }
}, './models/raw_dnn.pkl')

# mutual learning

## load models

In [221]:
train_df= pd.read_csv('./DS/iec104_train.csv', low_memory=False)
X_train = train_df[FS]
y_train = train_df['Label']

test_df= pd.read_csv('./DS/iec104_test.csv', low_memory=False)
X_test = test_df[FS]
y_test = test_df['Label']

In [234]:
xgb = joblib.load('./models/raw_xgb.pkl')
cat = joblib.load('./models/raw_cat.pkl')
rf = joblib.load('./models/raw_rf.pkl')
et = joblib.load('./models/raw_et.pkl')
lgbm = joblib.load('./models/raw_lgbm.pkl')

In [229]:
# DNN model
ckpt = torch.load("./models/raw_dnn.pkl", map_location=device)
dnn = ResDNN(
    in_dim=ckpt["config"]["num_features"],
    n_classes=ckpt["config"]["num_classes"]
).to(device)

dnn.load_state_dict(ckpt["model_state"])
dnn.eval()
scaler = ckpt["scaler"]
X_train_tensor = torch.tensor(scaler.transform(X_train), dtype=torch.float32).to(device)

X_test_scaled = scaler.transform(X_test)
# Chuyển sang tensor
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.long).to(device)
with torch.no_grad():
    probs = torch.softmax(dnn(X_test_tensor), dim=1)
    preds = probs.argmax(dim=1)
dnn_preds = probs.cpu().numpy()
dnn_prediction = preds.cpu().numpy()


# biLSTM
ckpt = torch.load("./models/raw_lstm.pkl", map_location=device)
lstm = LSTMTabular(
    step_dim=ckpt["config"]["step_dim"],
    n_classes=ckpt["config"]["num_classes"]
).to(device)

lstm.load_state_dict(ckpt["model_state"])
lstm.eval()
scaler = ckpt["scaler"]
X_test_scaled = scaler.transform(X_test)

# Chuyển sang tensor
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.long).to(device)
with torch.no_grad():
    logits = lstm(X_test_tensor)
    probs = torch.softmax(logits, dim=1)
    preds = probs.argmax(dim=1)
lstm_preds = probs.cpu().numpy()
lstm_prediction = preds.cpu().numpy()

/tmp/ipykernel_319864/2405897700.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load("./models/raw_dnn.pkl", map_location=device)
/home/soc/miniconda3/envs

## ensemble

In [195]:
# --- Predict proba ---
xgb_preds = xgb.predict_proba(X_test)
cat_preds = cat.predict_proba(X_test)
rf_preds  = rf.predict_proba(X_test)
et_preds  = et.predict_proba(X_test)
lgbm_preds = lgbm.predict_proba(X_test)

# --- Predict labels ---
xgb_prediction = xgb.predict(X_test)
cat_prediction = cat.predict(X_test)
rf_prediction  = rf.predict(X_test)
et_prediction  = et.predict(X_test)
lgbm_prediction = lgbm.predict(X_test)

In [62]:
# ====== cấu hình ======
metric = "f1"      # "f1" hoặc "recall"
step = 0.05
N = int(round(1/step))

# ====== danh sách proba của 6 mô hình (ví dụ: xgb, cat, rf, et, dnn, ftt) ======
preds_list = [xgb_preds, cat_preds, rf_preds, lgbm_preds, et_preds, lstm_preds] # , dnn_preds

best_score = -1.0
best_weights = None

# ====== search weights cho 6 mô hình ======
# w1 = i/N, w2 = j/N, w3 = k/N, w4 = l/N, w5 = m/N, w6 = n/N
for i in range(N + 1):
    for j in range(N + 1 - i):
        for k in range(N + 1 - i - j):
            for l in range(N + 1 - i - j - k):
                for m in range(N + 1 - i - j - k - l):
                    n = N - i - j - k - l - m   # n >= 0, đảm bảo tổng = N

                    w1, w2, w3, w4, w5, w6 = i/N, j/N, k/N, l/N, m/N, n/N

                    proba = (w1*preds_list[0] + w2*preds_list[1] +
                             w3*preds_list[2] + w4*preds_list[3] +
                             w5*preds_list[4] + w6*preds_list[5])

                    pred = proba.argmax(axis=1)

                    if metric == "f1":
                        score = f1_score(y_test, pred, average='macro', zero_division=0)
                    else:
                        score = recall_score(y_test, pred, average='macro', zero_division=0)

                    if score > best_score:
                        best_score = score
                        best_weights = (w1, w2, w3, w4, w5, w6)

print("Best weights (w1..w6):", best_weights, "sum =", sum(best_weights))
print(f"Best {metric} (macro):", best_score)

# ====== AUC macro OVR tại trọng số tốt nhất ======
w1, w2, w3, w4, w5, w6 = best_weights
proba_best = (w1*preds_list[0] + w2*preds_list[1] +
              w3*preds_list[2] + w4*preds_list[3] +
              w5*preds_list[4] + w6*preds_list[5])
y_pred = proba_best.argmax(axis=1)
print("Accuracy:", sklearn.metrics.accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision:", sklearn.metrics.precision_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("DR:", sklearn.metrics.recall_score(y_true=y_test, y_pred=y_pred, average='macro'))
print("F1:", sklearn.metrics.f1_score(y_true=y_test, y_pred=y_pred, average='macro'))
# labels = sorted(y_test.unique())
cm = sklearn.metrics.confusion_matrix(y_test, y_pred)
print("     " + " ".join(f"{lbl:6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{lbl:3d} " + " ".join(f"{val:6d}" for val in row))

Best weights (w1..w6): (0.0, 0.05, 0.15, 0.0, 0.05, 0.75) sum = 1.0
Best f1 (macro): 0.8544781190948494
Accuracy: 0.8589743589743589
Precision: 0.8554752603909188
DR: 0.858974358974359
F1: 0.8544781190948494
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      0    169      0      0      0      0      0      0      0      0      0
  3      0      0      0    169      0      0      0      0      0      0      0      0
  4      0      0      0      0    169      0      0      0      0      0      0      0
  5      0      0      0      0      0    169      0      0      0      0      0      0
  6      0      0      0      0      0      0    169      0      0      0      0      0
  7      0      1     37      0      0      0      0     75     38      7     11      0

## fix

In [ ]:
import numpy as np
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import label_binarize

# ====== CHỌN MỤC TIÊU ======
# 'logloss' -> khuyến nghị; 'auc' -> macro OvR
objective = 'auc'   # hoặc 'auc'

# ====== INPUTS ======
# Có sẵn 7 dự đoán xác suất:
preds_list = [xgb_preds, cat_preds, rf_preds, lgbm_preds, et_preds, lstm_preds, dnn_preds]

# Nhãn thật (n_samples,), dùng tập validation để tối ưu
y_val = y_test  # thay bằng biến của bạn; nếu chỉ có y_test, tạm dùng y_test (ít khuyến nghị hơn)

# ====== CHUẨN HOÁ SHAPE & HÀNG TỔNG = 1 ======
_fixed = []
for p in preds_list:
    p = np.asarray(p)
    if p.ndim == 1:
        # binary: vector prob class1 -> (n,2)
        p = np.column_stack([1.0 - p, p])
    elif p.ndim == 2 and p.shape[1] == 1:
        # binary: (n,1) -> (n,2)
        p = np.column_stack([1.0 - p[:, 0], p[:, 0]])
    # normalize mỗi hàng
    row_sum = p.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1.0
    p = p / row_sum
    _fixed.append(p)
preds_list = _fixed

n_classes = preds_list[0].shape[1]
for idx, p in enumerate(preds_list):
    assert p.shape[1] == n_classes, f"Model {idx} có C={p.shape[1]} khác {n_classes}"

# One-hot cho AUC macro nếu multiclass
if n_classes > 2:
    Y_bin = label_binarize(y_val, classes=np.arange(n_classes))

# ====== HÀM PHỤ TRỢ (projection lên simplex) ======
def project_simplex(v):
    """
    Euclidean projection of v onto the probability simplex {w | w>=0, sum(w)=1}
    """
    v = np.asarray(v, dtype=float)
    if v.sum() == 1.0 and np.all(v >= 0):
        return v
    # Thuật toán: sort giảm, tìm theta
    u = np.sort(v)[::-1]
    cssv = np.cumsum(u)
    rho = np.nonzero(u * np.arange(1, len(u)+1) > (cssv - 1))[0][-1]
    theta = (cssv[rho] - 1.0) / (rho + 1)
    w = np.maximum(v - theta, 0.0)
    return w

# ====== HÀM ĐÁNH GIÁ THEO MỤC TIÊU ======
def evaluate_objective(weights):
    # kết hợp xác suất
    proba = np.zeros_like(preds_list[0])
    for w, p in zip(weights, preds_list):
        proba += w * p
    if objective == 'logloss':
        return log_loss(y_val, proba), proba  # minimize
    else:  # 'auc'
        if n_classes == 2:
            auc = roc_auc_score(y_val, proba[:, 1])
        else:
            auc = roc_auc_score(Y_bin, proba, average='macro', multi_class='ovr')
        return -auc, proba  # minimize -AUC

# ====== 1) TỐI ƯU LIÊN TỤC: LOGLOSS BẰNG PROJECTED GRADIENT DESCENT ======
best_weights = None
best_obj = np.inf
best_proba = None

if objective == 'logloss':
    # PGD cấu hình
    rng = np.random.default_rng(42)
    restarts = 8            # số khởi tạo ngẫu nhiên
    max_iter = 800          # số vòng lặp PGD
    lr = 0.5                # learning rate ban đầu
    lr_decay = 0.995        # giảm LR mỗi vòng
    eps = 1e-12             # tránh chia 0

    # Khởi tạo thêm từ Dirichlet (đều, thiên về mô hình mạnh)
    inits = [np.ones(7)/7,
             rng.dirichlet(alpha=np.ones(7)*1.0),
             rng.dirichlet(alpha=np.array([3,2,2,3,2,1,2], dtype=float)),
             rng.dirichlet(alpha=np.array([4,1,1,4,1,1,3], dtype=float))]
    while len(inits) < restarts:
        inits.append(rng.dirichlet(alpha=np.ones(7)))

    for w0 in inits:
        w = w0.copy()
        cur_lr = lr
        # PGD loop
        for t in range(max_iter):
            # ensemble xác suất
            proba = np.zeros_like(preds_list[0])
            for wi, pi in zip(w, preds_list):
                proba += wi * pi
            # lấy proba của lớp đúng cho từng mẫu
            # idx: vector chỉ số lớp đúng
            idx = np.asarray(y_val, dtype=int)
            p_y = proba[np.arange(proba.shape[0]), idx]  # (n,)
            # gradient với mỗi weight i: d/dw_i [-1/n sum log p_y] = -1/n sum (p_i_y / p_y)
            g = np.zeros_like(w)
            for i in range(len(w)):
                p_i_y = preds_list[i][np.arange(proba.shape[0]), idx]
                g[i] = - np.mean(p_i_y / np.clip(p_y, eps, None))
            # bước cập nhật
            w = w - cur_lr * g
            # project lên simplex
            w = project_simplex(w)
            # giảm lr
            cur_lr *= lr_decay

            # early check mỗi 50 bước
            if (t+1) % 50 == 0:
                obj, proba_eval = evaluate_objective(w)
                if obj < best_obj:
                    best_obj = obj
                    best_weights = w.copy()
                    best_proba = proba_eval.copy()

    print("[LOGLOSS] Best weights:", best_weights, "sum =", best_weights.sum())
    print("[LOGLOSS] Best LogLoss:", best_obj)

# ====== 2) TỐI ƯU LIÊN TỤC: AUC BẰNG DIRICHLET RANDOM SEARCH + COORDINATE ASCENT ======
if objective == 'auc':
    rng = np.random.default_rng(123)
    samples = 4000              # số mẫu Dirichlet
    alphas = [
        np.ones(7)*1.0,         # đều
        np.array([3,2,2,3,2,1,2], dtype=float),  # ưu tiên xgb/lgbm/dnn chút
        np.array([4,1,1,4,1,1,3], dtype=float)
    ]
    best_obj = np.inf
    best_weights = None
    best_proba = None

    # Dirichlet sampling
    for alpha in alphas:
        for _ in range(samples // len(alphas)):
            w = rng.dirichlet(alpha)
            obj, proba = evaluate_objective(w)  # obj = -AUC
            if obj < best_obj:
                best_obj = obj
                best_weights = w.copy()
                best_proba = proba.copy()

    # Coordinate ascent tinh chỉnh quanh nghiệm tốt nhất
    delta = 0.10
    min_delta = 0.005
    patience = 0
    max_patience = 10
    while delta >= min_delta and patience <= max_patience:
        improved = False
        for i in range(7):
            w = best_weights.copy()
            # thử tăng weight i
            inc = min(delta, 1.0 - w[i])
            if inc > 0:
                w[i] += inc
                # giảm đều từ các trọng số khác (giữ sum=1, w>=0)
                mask = np.ones(7, dtype=bool); mask[i] = False
                dec_pool = w[mask].sum()
                if dec_pool > 0:
                    w[mask] = np.maximum(w[mask] * (1 - inc/dec_pool), 0.0)
                w = project_simplex(w)
                obj_inc, proba_inc = evaluate_objective(w)
            else:
                obj_inc = np.inf

            # thử giảm weight i
            w2 = best_weights.copy()
            dec = min(delta, w2[i])
            if dec > 0:
                w2[i] -= dec
                mask2 = np.ones(7, dtype=bool); mask2[i] = False
                w2[mask2] = project_simplex(w2[mask2] / w2[mask2].sum()) * (1 - w2[i])
                obj_dec, proba_dec = evaluate_objective(w2)
            else:
                obj_dec = np.inf

            # chọn cải thiện tốt hơn
            if obj_inc < best_obj or obj_dec < best_obj:
                if obj_inc <= obj_dec:
                    best_obj = obj_inc
                    best_weights = w
                    best_proba = proba_inc
                else:
                    best_obj = obj_dec
                    best_weights = w2
                    best_proba = proba_dec
                improved = True

        if improved:
            patience = 0
        else:
            patience += 1
            delta *= 0.7  # giảm bước tinh chỉnh

    print("[AUC] Best weights:", best_weights, "sum =", best_weights.sum())
    print("[AUC] Best AUC macro OvR:", -best_obj)  # vì obj = -AUC

# ====== ĐÁNH GIÁ CHI TIẾT TẠI TRỌNG SỐ TỐT NHẤT ======
proba_best = best_proba
y_pred = proba_best.argmax(axis=1)

print("Accuracy:", accuracy_score(y_true=y_val, y_pred=y_pred))
print("Precision (macro):", precision_score(y_true=y_val, y_pred=y_pred, average='macro', zero_division=0))
print("Recall (macro):",   recall_score(y_true=y_val, y_pred=y_pred, average='macro', zero_division=0))
print("F1 (macro):",       f1_score(y_true=y_val, y_pred=y_pred, average='macro', zero_division=0))

cm = confusion_matrix(y_val, y_pred, labels=labels)
print("\nConfusion matrix (rows=true, cols=pred):")
print("     " + " ".join(f"{int(lbl):6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{int(lbl):3d} " + " ".join(f"{int(val):6d}" for val in row))


[AUC] Best weights: [2.43077618e-02 2.93931005e-02 3.27596417e-01 1.77395268e-17
 1.46003228e-01 3.97577696e-02 4.32941723e-01] sum = 1.0
[AUC] Best AUC macro OvR: 0.989853455602204
Accuracy: 0.8717948717948718
Precision (macro): 0.8702682604174651
Recall (macro): 0.8717948717948718
F1 (macro): 0.8705171410436859

Confusion matrix (rows=true, cols=pred):
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      1    164      0      0      0      0      4      0      0      0      0
  3      0      0      0    164      0      4      1      0      0      0      0      0
  4      0      0      0      2    162      1      4      0      0      0      0      0
  5      0      0      0      0      1    168      0      0      0      0      0      0
  6      0      0      0  

## dynamic

In [196]:
import numpy as np
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import label_binarize

# ====== CHỌN MỤC TIÊU ======
# 'logloss' -> khuyến nghị; 'auc' -> macro OvR
objective = 'auc'   # hoặc 'logloss'

# ====== INPUTS ======
# Chỉ dùng 5 dự đoán: XGB, ExtraTree, RF, DNN, BiLSTM (thứ tự quan trọng để tùy chỉnh alpha nếu muốn)
preds_list = [xgb_preds, et_preds, rf_preds, dnn_preds, lstm_preds]
preds_list = [et_preds, dnn_preds]
# Nhãn thật (n_samples,), dùng tập validation để tối ưu
y_val = y_test  # thay bằng biến của bạn

# ====== CHUẨN HOÁ SHAPE & HÀNG TỔNG = 1 ======
_fixed = []
for p in preds_list:
    p = np.asarray(p)
    if p.ndim == 1:
        # binary: vector prob class1 -> (n,2)
        p = np.column_stack([1.0 - p, p])
    elif p.ndim == 2 and p.shape[1] == 1:
        # binary: (n,1) -> (n,2)
        p = np.column_stack([1.0 - p[:, 0], p[:, 0]])
    # normalize mỗi hàng
    row_sum = p.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1.0
    p = p / row_sum
    _fixed.append(p)
preds_list = _fixed

M = len(preds_list)      # số mô hình thực tế (here = 5)
n_classes = preds_list[0].shape[1]
for idx, p in enumerate(preds_list):
    assert p.shape[1] == n_classes, f"Model {idx} có C={p.shape[1]} khác {n_classes}"

# One-hot cho AUC macro nếu multiclass
if n_classes > 2:
    Y_bin = label_binarize(y_val, classes=np.arange(n_classes))

# Nếu 'labels' chưa có, đặt mặc định
try:
    labels
except NameError:
    labels = np.arange(n_classes)

# ====== HÀM PHỤ TRỢ (projection lên simplex) ======
def project_simplex(v):
    """
    Euclidean projection of v onto the probability simplex {w | w>=0, sum(w)=1}
    """
    v = np.asarray(v, dtype=float)
    if np.all(v >= 0) and abs(v.sum() - 1.0) < 1e-12:
        return v
    u = np.sort(v)[::-1]
    cssv = np.cumsum(u)
    rho_arr = u * np.arange(1, len(u) + 1) > (cssv - 1)
    if not np.any(rho_arr):
        # fallback: project to uniform
        return np.ones_like(v) / len(v)
    rho = np.nonzero(rho_arr)[0][-1]
    theta = (cssv[rho] - 1.0) / (rho + 1)
    w = np.maximum(v - theta, 0.0)
    return w

# ====== HÀM ĐÁNH GIÁ THEO MỤC TIÊU ======
def evaluate_objective(weights):
    # kết hợp xác suất
    proba = np.zeros_like(preds_list[0])
    for w, p in zip(weights, preds_list):
        proba += w * p
    if objective == 'logloss':
        return log_loss(y_val, proba), proba  # minimize
    else:  # 'auc'
        if n_classes == 2:
            auc = roc_auc_score(y_val, proba[:, 1])
        else:
            auc = roc_auc_score(Y_bin, proba, average='macro', multi_class='ovr')
        return -auc, proba  # minimize -AUC

# ====== 1) TỐI ƯU LIÊN TỤC: LOGLOSS BẰNG PROJECTED GRADIENT DESCENT ======
best_weights = None
best_obj = np.inf
best_proba = None

if objective == 'logloss':
    # PGD cấu hình
    rng = np.random.default_rng(42)
    restarts = 8            # số khởi tạo ngẫu nhiên
    max_iter = 800          # số vòng lặp PGD
    lr = 0.5                # learning rate ban đầu
    lr_decay = 0.995        # giảm LR mỗi vòng
    eps = 1e-12             # tránh chia 0

    # Khởi tạo thêm từ Dirichlet (đều, thiên về mô hình mạnh)
    # Tùy chỉnh alpha vector có độ dài M
    inits = [np.ones(M)/M,
             rng.dirichlet(alpha=np.ones(M)*1.0),
             rng.dirichlet(alpha=np.array([3 if i==0 else 2 for i in range(M)], dtype=float)),
             rng.dirichlet(alpha=np.array([4 if i==0 else 1 for i in range(M)], dtype=float))]
    while len(inits) < restarts:
        inits.append(rng.dirichlet(alpha=np.ones(M)))

    for w0 in inits:
        w = w0.copy()
        cur_lr = lr
        # PGD loop
        for t in range(max_iter):
            # ensemble xác suất
            proba = np.zeros_like(preds_list[0])
            for wi, pi in zip(w, preds_list):
                proba += wi * pi
            # lấy proba của lớp đúng cho từng mẫu
            idx = np.asarray(y_val, dtype=int)
            p_y = proba[np.arange(proba.shape[0]), idx]  # (n,)
            # gradient với mỗi weight i: d/dw_i [-1/n sum log p_y] = -1/n sum (p_i_y / p_y)
            g = np.zeros_like(w)
            for i in range(len(w)):
                p_i_y = preds_list[i][np.arange(proba.shape[0]), idx]
                g[i] = - np.mean(p_i_y / np.clip(p_y, eps, None))
            # bước cập nhật
            w = w - cur_lr * g
            # project lên simplex
            w = project_simplex(w)
            # giảm lr
            cur_lr *= lr_decay

            # early check mỗi 50 bước
            if (t+1) % 50 == 0:
                obj, proba_eval = evaluate_objective(w)
                if obj < best_obj:
                    best_obj = obj
                    best_weights = w.copy()
                    best_proba = proba_eval.copy()

    print("[LOGLOSS] Best weights:", best_weights, "sum =", best_weights.sum())
    print("[LOGLOSS] Best LogLoss:", best_obj)

# ====== 2) TỐI ƯU LIÊN TỤC: AUC BẰNG DIRICHLET RANDOM SEARCH + COORDINATE ASCENT ======
if objective == 'auc':
    rng = np.random.default_rng(123)
    samples = 4000              # số mẫu Dirichlet
    # Danh sách alphas (mỗi mục có chiều M)
    alphas = [
        np.ones(M)*1.0,         # đều
        # ưu tiên XGB (index 0) và RF (index 2) & DNN (index 3) ví dụ
        np.array([3 if i==0 else (2 if i in (2,3) else 1) for i in range(M)], dtype=float),
        np.array([4 if i==0 else (3 if i==2 else 1) for i in range(M)], dtype=float)
    ]
    best_obj = np.inf
    best_weights = None
    best_proba = None

    # Dirichlet sampling
    for alpha in alphas:
        for _ in range(samples // len(alphas)):
            w = rng.dirichlet(alpha)
            obj, proba = evaluate_objective(w)  # obj = -AUC
            if obj < best_obj:
                best_obj = obj
                best_weights = w.copy()
                best_proba = proba.copy()

    # Coordinate ascent tinh chỉnh quanh nghiệm tốt nhất
    delta = 0.10
    min_delta = 0.005
    patience = 0
    max_patience = 10
    while delta >= min_delta and patience <= max_patience:
        improved = False
        for i in range(M):
            w = best_weights.copy()
            # thử tăng weight i
            inc = min(delta, 1.0 - w[i])
            if inc > 0:
                w[i] += inc
                # giảm đều từ các trọng số khác (giữ sum=1, w>=0)
                mask = np.ones(M, dtype=bool); mask[i] = False
                dec_pool = w[mask].sum()
                if dec_pool > 0:
                    w[mask] = np.maximum(w[mask] * (1 - inc/dec_pool), 0.0)
                w = project_simplex(w)
                obj_inc, proba_inc = evaluate_objective(w)
            else:
                obj_inc = np.inf
                proba_inc = None

            # thử giảm weight i
            w2 = best_weights.copy()
            dec = min(delta, w2[i])
            if dec > 0:
                w2[i] -= dec
                mask2 = np.ones(M, dtype=bool); mask2[i] = False
                # renormalize the remaining to sum to (1 - w2[i])
                rem = w2[mask2]
                rem_sum = rem.sum()
                if rem_sum > 0:
                    rem = rem / rem_sum * (1.0 - w2[i])
                w2[mask2] = rem
                w2 = project_simplex(w2)
                obj_dec, proba_dec = evaluate_objective(w2)
            else:
                obj_dec = np.inf
                proba_dec = None

            # chọn cải thiện tốt hơn
            if obj_inc < best_obj or obj_dec < best_obj:
                if obj_inc <= obj_dec:
                    best_obj = obj_inc
                    best_weights = w
                    best_proba = proba_inc
                else:
                    best_obj = obj_dec
                    best_weights = w2
                    best_proba = proba_dec
                improved = True

        if improved:
            patience = 0
        else:
            patience += 1
            delta *= 0.7  # giảm bước tinh chỉnh

    print("[AUC] Best weights:", best_weights, "sum =", best_weights.sum())
    print("[AUC] Best AUC macro OvR:", -best_obj)  # vì obj = -AUC

# ====== ĐÁNH GIÁ CHI TIẾT TẠI TRỌNG SỐ TỐT NHẤT ======
proba_best = best_proba
y_pred = proba_best.argmax(axis=1)

print("Accuracy:", accuracy_score(y_true=y_val, y_pred=y_pred))
print("Precision (macro):", precision_score(y_true=y_val, y_pred=y_pred, average='macro', zero_division=0))
print("Recall (macro):",   recall_score(y_true=y_val, y_pred=y_pred, average='macro', zero_division=0))
print("F1 (macro):",       f1_score(y_true=y_val, y_pred=y_pred, average='macro', zero_division=0))

cm = confusion_matrix(y_val, y_pred, labels=labels)
print("\nConfusion matrix (rows=true, cols=pred):")
print("     " + " ".join(f"{int(lbl):6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{int(lbl):3d} " + " ".join(f"{int(val):6d}" for val in row))


[AUC] Best weights: [0.44422674 0.55577326] sum = 1.0
[AUC] Best AUC macro OvR: 0.9876383667917578
Accuracy: 0.8520710059171598
Precision (macro): 0.8497567363325031
Recall (macro): 0.8520710059171598
F1 (macro): 0.8477956449501867

Confusion matrix (rows=true, cols=pred):
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      0    169      0      0      0      0      0      0      0      0      0
  3      0      0      0    169      0      0      0      0      0      0      0      0
  4      0      0      0      0    169      0      0      0      0      0      0      0
  5      0      0      0      0      1    168      0      0      0      0      0      0
  6      0      0      0      0      0      0    169      0      0      0      0      0
  7      0      1    

## weighted combination

In [212]:
et_classes = [0,1,2,3,4,5,6,11]
dnn_classes = [7,8,10,9]

y_pred_final = []

for i in range(len(et_prediction)):
    # Nếu ET dự đoán vào lớp đáng tin et_classes
    if et_prediction[i] in et_classes:
        y_pred_final.append(et_prediction[i])
    else:
        # Còn lại (7–10) thì dùng DNN
        y_pred_final.append(dnn_prediction[i])

y_pred = np.array(y_pred_final)
print("Accuracy:", accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision (macro):", precision_score(y_true=y_test, y_pred=y_pred, average='macro', zero_division=0))
print("Recall (macro):",   recall_score(y_true=y_test, y_pred=y_pred, average='macro', zero_division=0))
print("F1 (macro):",       f1_score(y_true=y_test, y_pred=y_pred, average='macro', zero_division=0))
cm = confusion_matrix(y_test, y_pred, labels=labels)
print("\nConfusion matrix (rows=true, cols=pred):")
print("     " + " ".join(f"{int(lbl):6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{int(lbl):3d} " + " ".join(f"{int(val):6d}" for val in row))

Accuracy: 0.881163708086785
Precision (macro): 0.8910151461113052
Recall (macro): 0.8811637080867851
F1 (macro): 0.8684474049577862

Confusion matrix (rows=true, cols=pred):
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      0    169      0      0      0      0      0      0      0      0      0
  3      0      0      0    169      0      0      0      0      0      0      0      0
  4      0      0      0      0    169      0      0      0      0      0      0      0
  5      0      0      0      0      0    169      0      0      0      0      0      0
  6      0      0      0      0      0      0    169      0      0      0      0      0
  7      0      1     39      0      0      0      0     63     66      0      0      0
  8      0      0      0      0  

In [208]:
alpha = 0.7 # độ tin tưởng vào ET

combined_proba = alpha * et_preds + (1 - alpha) * dnn_preds
# combined_proba = et_preds

# Sau đó vẫn có thể override theo nhóm lớp như trên nếu muốn chắc chắn hơn
combined_proba[:, et_classes] = et_preds[:, et_classes]
combined_proba[:, dnn_classes] = (1 - alpha) * dnn_preds[:, dnn_classes]
# combined_proba[:, dnn_classes] = dnn_preds[:, dnn_classes]

y_pred = np.argmax(combined_proba, axis=1)
print("Accuracy:", accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision (macro):", precision_score(y_true=y_test, y_pred=y_pred, average='macro', zero_division=0))
print("Recall (macro):",   recall_score(y_true=y_test, y_pred=y_pred, average='macro', zero_division=0))
print("F1 (macro):",       f1_score(y_true=y_test, y_pred=y_pred, average='macro', zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=labels)
print("\nConfusion matrix (rows=true, cols=pred):")
print("     " + " ".join(f"{int(lbl):6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{int(lbl):3d} " + " ".join(f"{int(val):6d}" for val in row))

Accuracy: 0.834319526627219
Precision (macro): 0.8097777731653147
Recall (macro): 0.834319526627219
F1 (macro): 0.7786844160526082

Confusion matrix (rows=true, cols=pred):
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      0    169      0      0      0      0      0      0      0      0      0
  3      0      0      0    169      0      0      0      0      0      0      0      0
  4      0      0      0      0    169      0      0      0      0      0      0      0
  5      0      0      0      0      0    169      0      0      0      0      0      0
  6      0      0      0      0      0      0    169      0      0      0      0      0
  7      0      1     53      0      0      0      1      0     76      0     38      0
  8      0      0      0      0   

In [209]:
# Lấy xác suất dự đoán
# dnn_preds = np.exp(dnn_preds) / np.sum(np.exp(dnn_preds), axis=1, keepdims=True)  # softmax

# Kết hợp xác suất
combined_proba = np.zeros_like(et_preds)

combined_proba[:, et_classes] = et_preds[:, et_classes]
combined_proba[:, dnn_classes] = dnn_preds[:, dnn_classes]

# Chuẩn hóa lại (đảm bảo tổng = 1)
combined_proba = combined_proba / combined_proba.sum(axis=1, keepdims=True)

# Dự đoán nhãn cuối cùng
y_pred = np.argmax(combined_proba, axis=1)
print("Accuracy:", accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision (macro):", precision_score(y_true=y_test, y_pred=y_pred, average='macro', zero_division=0))
print("Recall (macro):",   recall_score(y_true=y_test, y_pred=y_pred, average='macro', zero_division=0))
print("F1 (macro):",       f1_score(y_true=y_test, y_pred=y_pred, average='macro', zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=labels)
print("\nConfusion matrix (rows=true, cols=pred):")
print("     " + " ".join(f"{int(lbl):6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{int(lbl):3d} " + " ".join(f"{int(val):6d}" for val in row))

Accuracy: 0.841715976331361
Precision (macro): 0.8439392817383289
Recall (macro): 0.8417159763313609
F1 (macro): 0.8139038115640013

Confusion matrix (rows=true, cols=pred):
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      0    169      0      0      0      0      0      0      0      0      0
  3      0      0      0    169      0      0      0      0      0      0      0      0
  4      0      0      0      0    169      0      0      0      0      0      0      0
  5      0      0      0      0      0    169      0      0      0      0      0      0
  6      0      0      0      0      0      0    169      0      0      0      0      0
  7      0      1     47      0      0      0      1     27     64      0     29      0
  8      0      0      0      0  

In [211]:
et_classes = [0,1,2,3,4,5,6,8,10,11]
dnn_classes = [7,9]

threshold_et_7  = 0.6   # ET chỉ giữ lớp 7 nếu tự tin >= 60%
threshold_dnn_7 = 0.5  # DNN chỉ ghi đè nếu tự tin > 55%

y_pred_final = []

for i in range(len(et_prediction)):
    et_label = et_prediction[i]
    dnn_label = dnn_prediction[i]
    p_et = et_preds[i]
    p_dnn = dnn_preds[i]

    if et_label in et_classes:
        # Trường hợp ET dự đoán lớp mạnh (0–6,8,11)
        y_pred_final.append(et_label)

    else:
        # Trường hợp ET ra lớp yếu (thường là 7,9,10)
        # ---- Lớp 7 đặc biệt: kiểm tra độ tự tin ----
        if dnn_label == 7:
            if p_et[7] >= threshold_et_7 and p_et[7] > p_dnn[7]:
                # ET vẫn rất tự tin vào 7 → giữ ET
                y_pred_final.append(7)
            elif p_dnn[7] >= threshold_dnn_7:
                # DNN đủ tự tin → nhận kết quả DNN
                y_pred_final.append(7)
            else:
                # Nếu cả hai không tự tin, chọn mô hình có xác suất cao hơn
                y_pred_final.append(7 if p_dnn[7] > p_et[7] else et_label)
        else:
            # Các lớp còn lại (9,10)
            y_pred_final.append(dnn_label)

y_pred = np.array(y_pred_final)

print("Accuracy:", accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision (macro):", precision_score(y_true=y_test, y_pred=y_pred, average='macro', zero_division=0))
print("Recall (macro):",   recall_score(y_true=y_test, y_pred=y_pred, average='macro', zero_division=0))
print("F1 (macro):",       f1_score(y_true=y_test, y_pred=y_pred, average='macro', zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=labels)
print("\nConfusion matrix (rows=true, cols=pred):")
print("     " + " ".join(f"{int(lbl):6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{int(lbl):3d} " + " ".join(f"{int(val):6d}" for val in row))


Accuracy: 0.8683431952662722
Precision (macro): 0.8782877428154378
Recall (macro): 0.8683431952662722
F1 (macro): 0.8499804612198485

Confusion matrix (rows=true, cols=pred):
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      0    169      0      0      0      0      0      0      0      0      0
  3      0      0      0    169      0      0      0      0      0      0      0      0
  4      0      0      0      0    169      0      0      0      0      0      0      0
  5      0      0      0      0      0    169      0      0      0      0      0      0
  6      0      0      0      0      0      0    169      0      0      0      0      0
  7      0      1     39      0      0      0      0     44     67      5     13      0
  8      0      0      0      0 

In [199]:
import numpy as np
from sklearn.metrics import (
    f1_score, recall_score, accuracy_score, precision_score, confusion_matrix
)

# ====== cấu hình ======
metric = "recall"      # "f1" hoặc "recall"
step = 0.01
N = int(round(1/step))

# ====== proba của 2 mô hình ======
preds_list = [et_preds, dnn_preds] # , dnn_preds

best_score = -1.0
best_weights = None

# ====== search weights cho 2 mô hình ======
for i in range(N + 1):
    w1 = i / N
    w2 = 1.0 - w1

    proba = w1 * preds_list[0] + w2 * preds_list[1]
    y_pred = proba.argmax(axis=1)

    if metric == "f1":
        score = f1_score(y_test, y_pred, average='macro', zero_division=0)
    else:
        score = recall_score(y_test, y_pred, average='macro', zero_division=0)

    if score > best_score:
        best_score = score
        best_weights = (w1, w2)

print("Best weights (w1, w2):", best_weights, "sum =", sum(best_weights))
print(f"Best {metric} (macro):", best_score)

# ====== đánh giá chi tiết tại trọng số tốt nhất ======
w1, w2 = best_weights
proba_best =proba = w1 * preds_list[0] + w2 * preds_list[1]
y_pred = proba_best.argmax(axis=1)

print("Accuracy:", accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision (macro):", precision_score(y_true=y_test, y_pred=y_pred, average='macro', zero_division=0))
print("Recall (macro):", recall_score(y_true=y_test, y_pred=y_pred, average='macro', zero_division=0))
print("F1 (macro):", f1_score(y_true=y_test, y_pred=y_pred, average='macro', zero_division=0))
cm = confusion_matrix(y_val, y_pred, labels=labels)
print("\nConfusion matrix (rows=true, cols=pred):")
print("     " + " ".join(f"{int(lbl):6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{int(lbl):3d} " + " ".join(f"{int(val):6d}" for val in row))

Best weights (w1, w2): (0.18, 0.8200000000000001) sum = 1.0
Best recall (macro): 0.8579881656804734
Accuracy: 0.8579881656804734
Precision (macro): 0.8552791091590893
Recall (macro): 0.8579881656804734
F1 (macro): 0.8535234623093467

Confusion matrix (rows=true, cols=pred):
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      0    169      0      0      0      0      0      0      0      0      0
  3      0      0      0    169      0      0      0      0      0      0      0      0
  4      0      0      0      0    169      0      0      0      0      0      0      0
  5      0      0      0      0      2    167      0      0      0      0      0      0
  6      0      0      0      0      0      0    169      0      0      0      0      0
  7      0      1   

## stacking

In [238]:
import numpy as np
import xgboost 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

X_train_tensor = torch.tensor(scaler.transform(X_train), dtype=torch.float32).to(device)
xgb_preds_train = xgb.predict_proba(X_train)
rf_preds_train  = rf.predict_proba(X_train)
et_preds_train  = et.predict_proba(X_train)
# dnn_preds_train = torch.softmax(dnn(X_train_tensor), dim=1).numpy()
# lstm_preds_train = torch.softmax(lstm(X_train_tensor), dim=1).numpy()

with torch.no_grad():
    dnn_probs = torch.softmax(dnn(X_train_tensor), dim=1)    
    lstm_probs = torch.softmax(lstm(X_train_tensor), dim=1)    
dnn_preds_train = dnn_probs.cpu().numpy()
lstm_preds_train = lstm_probs.cpu().numpy()

# Stack các xác suất dự đoán từ các mô hình
# meta_features = np.hstack([xgb_preds, rf_preds, et_preds, dnn_preds, lstm_preds])
# meta_features = np.hstack([et_preds, dnn_preds])
# meta_train = np.hstack([et_train, dnn_train])
meta_train = np.hstack([xgb_preds_train, rf_preds_train, et_preds_train, dnn_preds_train, lstm_preds_train])
meta_test  = np.hstack([xgb_preds, rf_preds, et_preds, dnn_preds, lstm_preds])


# # Huấn luyện Logistic Regression làm meta-classifier
# meta_clf = LogisticRegression(max_iter=1000)
# meta_clf.fit(meta_features, y_test)

# # Dự đoán nhãn cuối cùng
# y_preds = meta_clf.predict(meta_features)


# Tạo DMatrix cho XGBoost
dtrain = xgboost.DMatrix(meta_train, label=y_train)
dtest  = xgboost.DMatrix(meta_test, label=y_test)


# Cấu hình XGBoost cho phân loại đa lớp
params = {
    'objective': 'multi:softmax',
    'tree_method': 'gpu_hist',         
    'predictor': 'gpu_predictor',  
    'num_class': et_preds.shape[1],
    'eval_metric': 'mlogloss',
    'verbosity': 0
}

# Huấn luyện mô hình meta
meta_model = xgboost.train(params, dtrain, num_boost_round=100)

# Dự đoán
y_pred = meta_model.predict(dtest)


print("Accuracy:", accuracy_score(y_true=y_test, y_pred=y_pred))
print("Precision (macro):", precision_score(y_true=y_test, y_pred=y_pred, average='macro', zero_division=0))
print("Recall (macro):", recall_score(y_true=y_test, y_pred=y_pred, average='macro', zero_division=0))
print("F1 (macro):", f1_score(y_true=y_test, y_pred=y_pred, average='macro', zero_division=0))
cm = confusion_matrix(y_val, y_pred, labels=labels)
print("\nConfusion matrix (rows=true, cols=pred):")
print("     " + " ".join(f"{int(lbl):6d}" for lbl in labels))
for lbl, row in zip(labels, cm):
    print(f"{int(lbl):3d} " + " ".join(f"{int(val):6d}" for val in row))

/home/soc/miniconda3/envs/soc/lib/python3.8/site-packages/sklearn/base.py:458: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


Accuracy: 0.8081854043392505
Precision (macro): 0.8183666781971956
Recall (macro): 0.8081854043392503
F1 (macro): 0.7943685476555545

Confusion matrix (rows=true, cols=pred):
          0      1      2      3      4      5      6      7      8      9     10     11
  0    169      0      0      0      0      0      0      0      0      0      0      0
  1      0    169      0      0      0      0      0      0      0      0      0      0
  2      0      0    169      0      0      0      0      0      0      0      0      0
  3      0      0      0    169      0      0      0      0      0      0      0      0
  4      0      0      0      0    169      0      0      0      0      0      0      0
  5      2      0      0      0      0    167      0      0      0      0      0      0
  6      0      0      0      0      0      0    169      0      0      0      0      0
  7     82      0     26      0      0      0      0     35     19      4      3      0
  8     19      0      0      0 